# Capture and look

**Rung 0 of 5 (next: rung 1, [02, Address and read](02_address_and_read.ipynb)). You are here: one
call, and reading what it prints.**
Trigger: you have a model in hand.

`tl.trace(model, x)` runs one forward pass and records every operation into a `Trace`. This
notebook is about that one call: what it returns, the input forms it accepts, the summary and its
views, and the promise that a capture leaves the model exactly as it found it.

> **You'll want to know:** how to build an `nn.Module` (PyTorch), how to call it on a tensor
> (PyTorch), and nothing else. If you would rather see the whole core first, run
> [00, TorchLens in five minutes](00_quickstart.ipynb).

In [ ]:
import tempfile
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [ ]:
from learn_models import tiny_cnn, tiny_mlp  # tiny seeded CPU models; nothing downloads

cnn, image = tiny_cnn()  # a stem, six conv blocks (conv, BatchNorm, ReLU), pooling, a linear head
model, x = tiny_mlp()  # two linear layers with a ReLU between them, and a (4, 8) input batch

One call captures the CNN. The full-depth summary folds its six identical blocks into one row,
so the whole model reads in seven lines: where the parameters and the FLOPs sit, module by
module.


In [ ]:
trace = tl.trace(cnn, image)
trace.summary(depth="all")

## 1. One verb

`tl.trace` returns a `Trace`. It holds a record for every operation the forward ran, grouped
into the modules they ran in; `trace.params` adds up the parameters and `trace.output` is what
the model returned. `print(trace)` is the long form, a full log of the pass; the counts are the
short one.

In [ ]:
print(trace.num_ops, "ops |", len(trace.modules), "modules |", trace.params.numel, "parameters")
tuple(trace.output.shape)

## 2. The summary and its views

A bare `trace.summary()` is the overview: one row per module or operation, at a depth chosen to
stay under 48 rows, with the output shape, the parameters and their share, and the forward FLOPs.
On this CNN that opens every block, 30 rows; run it when you want them. Three dials change the
grain. `depth="all"` (the cell above) shows the module tree to its full depth and folds repeated
siblings into one row (`0..5 (ConvBlock x6)`), so it can be shorter than the default.
`view="compute"` adds MACs and an evidence column saying how each count was obtained.
`level="op"` goes down to one row per operation, labelled the way `trace[...]` addresses them;
on the two-layer MLP that is three rows. `str()` of any summary is plain ASCII text you can
paste anywhere.


In [ ]:
trace.summary(view="compute", depth="all")

In [ ]:
tl.trace(model, x).summary(level="op")  # the MLP: one row per operation

## 3. Three ways to hand over an input

A real input is the best one: the shapes, the values and the path through `forward` are all
yours. Without one, pass `input_shape=` and TorchLens synthesizes seeded random values and says so
in the summary. With nothing at all it infers a shape and discloses that too, or tells you what
it needs. All three agree on the structure. A `forward` that takes keyword arguments gets them
through `input_kwargs=`. The cell reads each trace's input record as `t["input_1"]`, a label in
square brackets; [02](02_address_and_read.ipynb) is about those labels.

In [ ]:
real = tl.trace(cnn, image)
shaped = tl.trace(cnn, input_shape=(2, 1, 16, 16))  # your shape, seeded random values, disclosed
inferred = tl.trace(cnn)  # nothing: TorchLens infers a shape (a standard image size here)
by_keyword = tl.trace(model, input_kwargs={"x": x})  # forward(self, x): the same tensor by name
for name, t in [("real input", real), ("input_shape", shaped), ("inferred", inferred)]:
    print(f"{name:>12}: {t.num_ops} ops, input {tuple(t['input_1'].shape)}")
print(f"{'by keyword':>12}: {by_keyword.num_ops} ops, input {tuple(by_keyword['input_1'].shape)}")

For a Hugging Face model a prompt string is a real input: `tl.trace(lm, "The quick brown fox")`
tokenizes with the model's own tokenizer. Nothing in this notebook downloads, so that line stays
text here; [08, Semantic shortcuts](08_semantic_shortcuts.ipynb) explains the prompt route.

## 4. The one-call doors

When the summary is all you want, skip the trace. `tl.summary(model, x)` runs a metadata-only
capture in eval mode under no-grad, restores every training flag afterwards and returns the
report. `tl.draw(model, x)` is the same door for a picture, in [03, See it](03_see_it.ipynb). Both
take the same three input forms as `tl.trace`.

In [ ]:
tl.summary(model, x)  # the MLP; the header says what the one-call door restored

## 5. A trace is a pure observer

A capture never changes the model. Put the CNN in train mode, where a forward pass updates every
BatchNorm's running statistics, and trace it: the statistics do not move. Then call the model
plainly once, and they do. The capture snapshots buffers, training flags and random states before
the forward and restores them after. The trace's `outcome` says the forward ran to completion,
and `is_pristine` says nothing intervened in it.

In [ ]:
stats = cnn.blocks[0].norm.running_mean  # a BatchNorm buffer that train-mode forwards update
cnn.train()
print("before             :", stats[:3].clone())
observed = tl.trace(cnn, image)
print("after tl.trace     :", stats[:3].clone())
cnn(image)  # a plain forward, for comparison
print("after a plain call :", stats[:3].clone())
cnn.eval()
print("outcome:", observed.outcome.status.name, "| pristine:", observed.is_pristine)

## 6. Where this leaves you

You can capture any model with one call, hand it an input four ways, read the summary at four
grains, and trust that the model is untouched afterwards. Everything from here is reading the
`Trace` more closely.

**One thing about networks:** parameters and compute live in different places. In the compute
view the stem's 80 parameters do 77.82K FLOPs, about a thousand each, because a convolution
reuses every weight at every pixel; the head's 90 parameters do 340, about four each, because a
linear layer uses each weight once per item. The six blocks hold 3.6K of the 3,770 parameters
and nearly all of the compute. Compute follows resolution, parameters follow width.

**Go deeper:** next rung, [02, Address and read](02_address_and_read.ipynb): every operation has
an address and a record behind it. Recipe card:
[`visualize.ipynb`](../../examples/5min/visualize.ipynb). Reference:
[the quick-start guide](../../docs/quickstart.md) for the input ladder and
[the summary reference](../../docs/reference/summary.md) for every column and dial.